# Лабораторная 06. Shuffle join vs broadcast join

Цель: сравнить SortMergeJoin и BroadcastHashJoin.

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast

spark = (SparkSession.builder.appName('lab-06-broadcast-join').master('local[*]')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'false')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
base_uri = Path('spark_core_data').absolute().as_uri()
orders = spark.read.parquet(f'{base_uri}/orders')
customers = spark.read.parquet(f'{base_uri}/customers')
print('Spark UI:', spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/07 10:40:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

Spark UI: http://0a370e2ebe67:4040


## Часть 1. Отключаем auto broadcast

In [2]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '-1')
shuffle_join = orders.join(customers, 'customer_id')
shuffle_join.explain('formatted')
shuffle_join.count()

== Physical Plan ==
* Project (12)
+- * SortMergeJoin Inner (11)
   :- * Sort (5)
   :  +- Exchange (4)
   :     +- * Filter (3)
   :        +- * ColumnarToRow (2)
   :           +- Scan parquet  (1)
   +- * Sort (10)
      +- Exchange (9)
         +- * Filter (8)
            +- * ColumnarToRow (7)
               +- Scan parquet  (6)


(1) Scan parquet 
Output [5]: [order_id#0L, customer_id#1L, order_date#2, status#3, order_amount#4]
Batched: true
Location: InMemoryFileIndex [file:/materials/seminar_04_spark_core/practice/spark_core_data/orders]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<order_id:bigint,customer_id:bigint,order_date:date,status:string,order_amount:decimal(10,2)>

(2) ColumnarToRow [codegen id : 1]
Input [5]: [order_id#0L, customer_id#1L, order_date#2, status#3, order_amount#4]

(3) Filter [codegen id : 1]
Input [5]: [order_id#0L, customer_id#1L, order_date#2, status#3, order_amount#4]
Condition : isnotnull(customer_id#1L)

(4) Exchange
Input [5]: [order

120000

## Часть 2. Явный broadcast маленького справочника

In [3]:
broadcast_join = orders.join(broadcast(customers), 'customer_id')
broadcast_join.explain('formatted')
broadcast_join.count()

== Physical Plan ==
* Project (9)
+- * BroadcastHashJoin Inner BuildRight (8)
   :- * Filter (3)
   :  +- * ColumnarToRow (2)
   :     +- Scan parquet  (1)
   +- BroadcastExchange (7)
      +- * Filter (6)
         +- * ColumnarToRow (5)
            +- Scan parquet  (4)


(1) Scan parquet 
Output [5]: [order_id#0L, customer_id#1L, order_date#2, status#3, order_amount#4]
Batched: true
Location: InMemoryFileIndex [file:/materials/seminar_04_spark_core/practice/spark_core_data/orders]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<order_id:bigint,customer_id:bigint,order_date:date,status:string,order_amount:decimal(10,2)>

(2) ColumnarToRow [codegen id : 2]
Input [5]: [order_id#0L, customer_id#1L, order_date#2, status#3, order_amount#4]

(3) Filter [codegen id : 2]
Input [5]: [order_id#0L, customer_id#1L, order_date#2, status#3, order_amount#4]
Condition : isnotnull(customer_id#1L)

(4) Scan parquet 
Output [4]: [customer_id#10L, customer_name#11, region#12, segment#13]
Batche

120000

Сравните планы:

| Вопрос | Ответ |
|---|---|
| Какой join был в первом плане? | SortMergeJoin Inner |
| Какой join был во втором плане? | BroadcastHashJoin Inner BuildRight |
| Где исчез или уменьшился shuffle? | во втором плане (есть только BroadcastExchange) |
| Почему маленький справочник можно broadcast'ить? | потому что он занимает мало места и его легко передавать на все executor-ы |
| Когда broadcast может навредить? | когда справочник большой; когда у executors мало памяти; когда broadcast повторяется много раз |

Подсказка: ищите `SortMergeJoin`, `BroadcastHashJoin`, `Exchange`, `BroadcastExchange`.

In [4]:
spark.stop()